# 04 — Qiskit Export

Export encoded data as **Qiskit `QuantumCircuit`** objects.

```bash
pip install quprep[qiskit]
```

This notebook requires the `[qiskit]` extra. If Qiskit is not installed, cells will skip gracefully.

In [ ]:
import numpy as np
import pandas as pd

try:
    import qiskit
    QISKIT_AVAILABLE = True
    print(f"Qiskit {qiskit.__version__} detected")
except ImportError:
    QISKIT_AVAILABLE = False
    print("Qiskit not found. Install with:  pip install quprep[qiskit]")

## 1. Angle encoding → QuantumCircuit

In [ ]:
if not QISKIT_AVAILABLE:
    print("Skipped — Qiskit not installed")
else:
    from quprep.encode.angle import AngleEncoder
    from quprep.export.qiskit_export import QiskitExporter

    exporter = QiskitExporter()
    vec = np.array([0.3, 1.1, 0.7, 2.0])

    enc = AngleEncoder(rotation="ry").encode(vec)
    qc = exporter.export(enc)

    print(type(qc))
    qc.draw(output="mpl")

## 2. Amplitude encoding → QuantumCircuit

Uses Qiskit's `StatePreparation` gate — this is the only encoder/framework combination that supports amplitude encoding.

In [ ]:
if not QISKIT_AVAILABLE:
    print("Skipped — Qiskit not installed")
else:
    from quprep.encode.amplitude import AmplitudeEncoder
    from quprep.export.qiskit_export import QiskitExporter

    exporter = QiskitExporter()
    unit_vec = np.array([0.5, 0.5, 0.5, 0.5])

    enc_amp = AmplitudeEncoder().encode(unit_vec)
    qc_amp = exporter.export(enc_amp)

    print(f"Circuit: {qc_amp.num_qubits} qubits, {qc_amp.depth()} depth")
    qc_amp.draw(output="mpl")

## 3. Basis encoding → QuantumCircuit

In [ ]:
if not QISKIT_AVAILABLE:
    print("Skipped — Qiskit not installed")
else:
    from quprep.encode.basis import BasisEncoder
    from quprep.export.qiskit_export import QiskitExporter

    exporter = QiskitExporter()
    bits = np.array([1.0, 0.0, 1.0, 1.0])

    enc_basis = BasisEncoder().encode(bits)
    qc_basis = exporter.export(enc_basis)

    print(f"Bit string: {bits.astype(int).tolist()}")
    qc_basis.draw(output="mpl")

## 4. Full pipeline → Qiskit

In [ ]:
if not QISKIT_AVAILABLE:
    print("Skipped — Qiskit not installed")
else:
    from quprep import Pipeline
    from quprep.encode.angle import AngleEncoder
    from quprep.export.qiskit_export import QiskitExporter

    df = pd.DataFrame(
        {
            "x1": [0.1, 0.5, 0.9],
            "x2": [0.4, 0.2, 0.8],
            "x3": [0.7, 0.6, 0.3],
        }
    )

    pipeline = Pipeline(
        encoder=AngleEncoder(rotation="rx"),
        exporter=QiskitExporter(),
    )
    result = pipeline.fit_transform(df)

    print(f"Encoded {len(result.circuits)} circuits")
    result.circuits[0].draw(output="mpl")

## 5. Using the circuit in a Qiskit backend

Once you have a `QuantumCircuit` you can run it on any Qiskit-compatible backend.

In [ ]:
if not QISKIT_AVAILABLE:
    print("Skipped — Qiskit not installed")
else:
    # Simulate locally with Qiskit's Aer (if installed)
    try:
        from qiskit import transpile
        from qiskit_aer import AerSimulator

        qc_measured = qc.copy()
        qc_measured.measure_all()

        sim = AerSimulator()
        transpiled = transpile(qc_measured, sim)
        job = sim.run(transpiled, shots=1024)
        counts = job.result().get_counts()
        print("Measurement counts:", counts)
    except ImportError:
        print("qiskit-aer not installed. Install with: pip install qiskit-aer")
        print("Circuit object is ready — pass it to any Qiskit-compatible backend.")